# Faq

## Questions
### 1 Explain the code below
### 2 Fix it
### 3 Propose improvements 

The goal:
Given a user's question, find the most similar question in the FAQ.

Imagine a company help page with 4 known questions. A user types their own wording, like:

"Ou est la liste des fournisseurs"

The code should work out that this is really asking:

"Où trouver la liste des fournisseurs?"

So it acts as a tiny search engine, or the start of a chatbot: the user asks in their own words, and it finds which FAQ entry they mean. Once the right question is found, you could show the answer attached to it.

The below code loads the sklearn's CountVectorizer and then initializes its object
Then we have a corpus of documents in french language one which we fit the countvectorizer object.


### 1. Explanation: Vectorization and Corpus Representation

Machine learning algorithms and linear algebraic similarity metrics operate on numerical vector spaces rather than raw natural language strings. 

`CountVectorizer` converts a collection of text documents into a sparse document-term matrix (Bag-of-Words model):
- **Fitting (`cv.fit`)**: Scans the corpus, applies default preprocessing (lowercasing, punctuation stripping, tokenizing on word boundaries `\b\w\w+\b`), and constructs a fixed vocabulary mapping each unique token to a distinct column index. For instance, `l'entreprise` is tokenized into `entreprise`, while single-letter tokens like `l` are discarded by default.
- **Transformation**: Encodes each document into a numerical vector where the value at index $j$ represents the frequency count of vocabulary term $j$ in that document.

In [1]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()

all_questions = [
    "Quel est le nom de l'entreprise Corcentric?",
    "Quelle est la date de création du BIL?",
    "Comment enregistrer une demande d'achat?",
    "Où trouver la liste des fournisseurs?"
]

cv.fit(all_questions);

Explanation:

In below code, we use numpy imports to perform dot products on vectors but before the dot product,
we pass a query to be transformed using count vectorizer object, here the transform function produces the count vectors. Each text becomes a row, and each vocabulary word is a column holding how many times that word appears in that text.

to_dense(): converts the sparse result into a normal dense grid, which is a 1 × 23 matrix.

Then the loop:
It turns the user's query into a vector of word counts. For each FAQ question it computes the dot product, which is the number of words the query and question share (weighted by counts). max(res) picks the highest score, so it returns the most similar FAQ question.

### Identified Issues in the Starter Code

1. **Input type mismatch in `transform()`**: `cv.transform()` expects an iterable collection of strings (like `[query]`), but was passed a raw string `query`.
2. **Slow iterative loop**: Running a Python `for` loop to compute individual dot products question-by-question adds unnecessary overhead. Vectorized matrix multiplication (`Q @ q_vect.T`) computes scores across the entire corpus in a single pass.
3. **Data structure bug in `max()`**: `np.dot` between two 2D sparse slices produced nested matrix objects inside the results list. Python's `max()` fails or behaves unpredictably when trying to sort tuples containing multidimensional matrix objects instead of plain numbers.

In [26]:
import numpy as np


res = []
query = "Ou est la liste des fournisseurs"
q_vect = cv.transform(query).todense()  
for q in all_questions:
    res.append((np.dot(q_vect, cv.transform([q])[0].T)[0,0],
                q))
               
    
max(res)

In [16]:
# Solution fix
# Avoid the loop and do direct dot product as it is less computation heavy
# First transform all questions to the same numeric form in which query will arrive


query = "Ou est la liste des fournisseurs"

Q = cv.transform(all_questions)              # vectorize the FAQ once
q_vect = cv.transform([query])               # list, not string
scores = (Q @ q_vect.T).toarray().ravel()    # plain numbers: [1 2 0 4]

best = scores.argmax() # get position index of best score
print(all_questions[best], scores[best])

Où trouver la liste des fournisseurs? 4


In [23]:
(Q @ q_vect.T).toarray() # we have 4 questions and this scores each question wrt the query asked by user based on the similarity of words

array([[1],
       [2],
       [0],
       [4]], dtype=int64)

The dot product multiplies the query's word counts by a FAQ question's word counts and sums them, which effectively counts the words they share; the FAQ question with the highest dot product is the closest match.

In [34]:
# Improvement

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Filler words that carry little meaning. They are thrown away before scoring,
# so sharing "le" or "est" with a question no longer counts as a match.
# ("ou" is here because "Où" becomes "ou" once accents are stripped.)
FRENCH_STOPWORDS = ["le","la","les","un","une","des","du","de","est","et","au","aux",
                    "en","que","qui","quoi","quel","quelle","quels","quelles","ou",
                    "comment","ce","cette","ces"]


class FAQMatcher:
    """Finds the FAQ question most similar to what a user typed."""

    def __init__(self, questions, stop_words=FRENCH_STOPWORDS):
        # Keep our own copy of the questions, in the same order as the matrix rows.
        self.questions = list(questions)

        # The vectorizer turns text into numbers. Settings:
        #  - strip_accents="unicode": "création" and "creation" are treated as the same word
        #  - stop_words: the filler words above are ignored
        #  - (default) lowercase: "BIL" and "bil" are the same word
        self.vectorizer = TfidfVectorizer(strip_accents="unicode", stop_words=stop_words)

        # fit_transform does two things in one go:
        #  1. fit:       learns the vocabulary and how rare each word is across the FAQ
        #  2. transform: converts each FAQ question into a row of weighted word scores
        # Result: a matrix with one row per question and one column per word.
        # Done once here, so it is not repeated for every user query.
        self.matrix = self.vectorizer.fit_transform(self.questions)

    def match(self, query, threshold=0.2):
        # Turn the user's query into numbers using the SAME vocabulary and weights.
        # It must be a list ([query]), not a bare string. Words the FAQ has never
        # seen are ignored.
        query_vector = self.vectorizer.transform([query])

        # Compare the query with every FAQ question at once.
        # Cosine similarity gives a score from 0 (nothing in common) to 1 (identical),
        # and is not biased toward longer questions.
        # ravel() flattens the result into a simple list of 4 scores.
        scores = cosine_similarity(query_vector, self.matrix).ravel()

        # argmax gives the POSITION of the highest score (not the score itself).
        # Because scores and questions are in the same order, that position
        # tells us which question won. On a tie, the first one wins.
        best = scores.argmax()

        # If even the best score is too low, admit there is no good match
        # instead of returning a wrong answer. Returns (None, score).
        # 0.2 is a starting guess; it should be tuned on real queries.
        if scores[best] < threshold:
            return None, float(scores[best])

        # Otherwise return the winning question and its score.
        # float() converts numpy's number type into a plain Python number.
        return self.questions[best], float(scores[best])


# Build the matcher once (this learns the vocabulary), then reuse it for every query.
matcher = FAQMatcher(all_questions)

print(matcher.match("Ou est la liste des fournisseurs"))

('Où trouver la liste des fournisseurs?', 0.816496580927726)


The score grows with text length and word repetition, not only with similarity. A long question containing many words has more chances to overlap with the query. It also treats every word equally, so sharing "la" counts the same as sharing "fournisseurs".

### Why TF-IDF and Cosine Similarity Work Better

- **Eliminating length bias**: A raw dot product grows with sentence length. A long FAQ question can easily get a higher score just because it has more words that accidentally overlap with the query. Cosine similarity divides by vector lengths, measuring the angle between texts on a normalized scale from 0 to 1 regardless of how verbose they are.
- **Penalizing filler words**: In a basic bag-of-words model, shared filler words like *le*, *la*, *est*, or *comment* carry the exact same weight as domain-specific terms. TF-IDF downweights words that appear everywhere and amplifies discriminative keywords like *fournisseurs*, *commande*, or *facture*.
- **Accents & thresholds**: Normalizing accents (`strip_accents="unicode"`) ensures *où* and *ou* match seamlessly. Adding a similarity threshold prevents the search engine from returning a false positive when a user asks something completely outside the FAQ's scope.